In [ ]:
%pip install -q git+https://github.com/huggingface/transformers.git

In [ ]:
%pip install -q timm matplotlib

In [ ]:
from huggingface_hub import hf_hub_download
from PIL import Image
import matplotlib.pyplot as plt
# Load the image
file_path = '3eec9544-151_table_1.jpg'
image = Image.open(file_path).convert("RGB")

# # Check if the image is already in a supported format, if not convert to PNG
# if image.format not in {'PNG', 'JPEG', 'JPEG2000', 'PBM', 'PGM', 'PPM', 'TIFF', 'BMP', 'GIF', 'WEBP'}:
#     print(f"Image format {image.format} is not supported, converting to PNG...")
#     image = image.convert("RGB")
#     image.save("temp_image.png")  # Save as a temporary PNG image
#     image = Image.open("temp_image.png")



# Resize the image (ensure assignment)
width, height = image.size
image = image.resize((int(width * 0.5), int(height * 0.5)))

# Make sure 'image' is in the correct format (it should be a PIL image)
if not isinstance(image, Image.Image):
    raise TypeError("Image is not a PIL Image object!")

else:
    # Plot the image
    plt.figure(figsize=(12, 8))  
    plt.imshow(image)
    plt.axis('off')  # Hide axes
    plt.title("This is the sample original image")
    plt.show()

In [ ]:
# Import the DetrFeatureExtractor class from the Transformers library
from transformers import DetrImageProcessor

# Create an instance of the DetrFeatureExtractor
feature_extractor = DetrImageProcessor()

# Use the feature extractor to encode the image
# 'image' should be the PIL image object that was obtained earlier
encoding = feature_extractor(image, return_tensors="pt")

# Get the keys of the encoding dictionary
keys = encoding.keys()
keys

In [ ]:
# Import the TableTransformerForObjectDetection class from the transformers library
from transformers import TableTransformerForObjectDetection

# Load the pre-trained Table Transformer model for object detection
model = TableTransformerForObjectDetection.from_pretrained("microsoft/table-transformer-detection")

In [ ]:
import torch

# Disable gradient computation for inference
with torch.no_grad():
    # Pass the encoded image through the model for inference
    # 'model' is the TableTransformerForObjectDetection model loaded previously
    # 'encoding' contains the encoded image features obtained using the DetrFeatureExtractor
    outputs = model(**encoding)

outputs

In [ ]:
import matplotlib.pyplot as plt

# Define colors for visualization
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

def plot_results(pil_img, scores, labels, boxes):
    # Create a figure for visualization
    plt.figure(figsize=(16, 10))
    
    # Display the PIL image
    plt.imshow(pil_img)
    
    # Get the current axis
    ax = plt.gca()
    
    # Repeat the COLORS list multiple times for visualization
    colors = COLORS * 100
    
    # Iterate through scores, labels, boxes, and colors for visualization
    for score, label, (xmin, ymin, xmax, ymax), c in zip(scores.tolist(), labels.tolist(), boxes.tolist(), colors):
        # Add a rectangle to the image for the detected object's bounding box
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=3))
        
        # Prepare the text for the label and score
        text = f'{model.config.id2label[label]}: {score:0.2f}'
        
        # Add the label and score text to the image
        ax.text(xmin, ymin, text, fontsize=15,
                bbox=dict(facecolor='yellow', alpha=0.5))
    
    # Turn off the axis
    plt.axis('off')
    plt.savefig('identify_borderless.png', dpi = 1000)
    # Display the visualization
    plt.show()

In [ ]:
# Get the original width and height of the image
width, height = image.size

# Post-process the object detection outputs using the feature extractor
results = feature_extractor.post_process_object_detection(outputs, threshold=0.7, target_sizes=[(height, width)])[0]

# Plot the visualization of the results
plot_results(image, results['scores'], results['labels'], results['boxes'])

In [ ]:
# Use the feature extractor to encode the resized image
encoding = feature_extractor(image, return_tensors="pt")

# Get the keys of the encoding dictionary
keys = encoding.keys()
keys

In [ ]:
# Import the TableTransformerForObjectDetection class from the transformers library
from transformers import TableTransformerForObjectDetection

# Load the pre-trained Table Transformer model for table structure recognition
model = TableTransformerForObjectDetection.from_pretrained("microsoft/table-transformer-structure-recognition")

with torch.no_grad():
  outputs = model(**encoding)

In [ ]:
# Create a list of target sizes for post-processing
# 'image.size[::-1]' swaps the width and height to match the target size format (height, width)
target_sizes = [image.size[::-1]]

# Post-process the object detection outputs using the feature extractor
# Use a threshold of 0.6 for confidence
results = feature_extractor.post_process_object_detection(outputs, threshold=0.6, target_sizes=target_sizes)[0]

# Plot the visualization of the results
plot_results(image, results['scores'], results['labels'], results['boxes'])

# Inference pipeline

In [ ]:
from PIL import Image, UnidentifiedImageError
import torch
import matplotlib.pyplot as plt
from transformers import DetrImageProcessor, TableTransformerForObjectDetection

class BorderlessTableDetection:
    def __init__(self, image_path, output_path, detection_model="microsoft/table-transformer-detection",
                 structure_model="microsoft/table-transformer-structure-recognition"):
        """
        Initializes the BorderlessTableDetection class with models, feature extractor, and image paths.
        
        @param image_path: The path of the image to process.
        @param output_path: The path to save the output image.
        @param detection_model: Pre-trained model to detect tables.
        @param structure_model: Pre-trained model to recognize the structure of tables.
        """
        self.image_path = image_path
        self.output_path = output_path
        try:
            self.feature_extractor = DetrImageProcessor()
            self.detection_model = TableTransformerForObjectDetection.from_pretrained(detection_model)
            self.structure_model = TableTransformerForObjectDetection.from_pretrained(structure_model)
        except Exception as e:
            print(f"Error loading models or feature extractor: {e}")
            raise
        
        self.image = None
        self.encoding = None

    def load_image(self):
        """
        Loads the image from the given path and resizes it.

        @return: None
        """
        try:
            self.image = Image.open(self.image_path).convert("RGB")
            width, height = self.image.size
            # self.image = self.image.resize((int(width * 0.5), int(height * 0.5)))
        except FileNotFoundError:
            print(f"Error: The file {self.image_path} was not found.")
            raise
        except UnidentifiedImageError:
            print(f"Error: Unable to identify the image file at {self.image_path}.")
            raise
        except Exception as e:
            print(f"Unexpected error while loading image: {e}")
            raise
    
    def extract_features(self):
        """
        Extracts the features of the image using the DetrFeatureExtractor.

        @return: None
        """
        try:
            self.encoding = self.feature_extractor(self.image, return_tensors="pt")
        except Exception as e:
            print(f"Error during feature extraction: {e}")
            raise
    
    def detect_tables(self, model_type="detection"):
        """
        Detects tables or table structures from the image using the appropriate model.

        @param model_type: Specifies whether to use the detection model or structure model. Defaults to "detection".
        @return: The output predictions from the model.
        """
        model = self.detection_model if model_type == "detection" else self.structure_model
        try:
            with torch.no_grad():
                outputs = model(**self.encoding)
            return outputs
        except Exception as e:
            print(f"Error during table detection: {e}")
            raise

    def post_process(self, outputs, threshold=0.7):
        """
        Post-processes the model outputs to apply a confidence threshold.

        @param outputs: The raw outputs from the model.
        @param threshold: The confidence threshold for filtering results. Defaults to 0.7.
        @return: Processed results containing scores, labels, and bounding boxes.
        """
        try:
            width, height = self.image.size
            results = self.feature_extractor.post_process_object_detection(outputs, threshold=threshold, 
                                                                          target_sizes=[(height, width)])[0]
            return results
        except Exception as e:
            print(f"Error during post-processing: {e}")
            raise

    def plot_results(self, results, scores, labels, boxes, model_type="detection"):
        """
        Plots the detected tables on the image and saves the result.

        @param results: The processed results from the model, including scores, labels, and bounding boxes.
        @param scores: Confidence scores for the detected tables.
        @param labels: Labels for the detected tables.
        @param boxes: Bounding boxes for the detected tables.
        @param model_type: Specifies which model's labels are being used (detection or structure). Defaults to "detection".
        @return: None
        """
        try:
            COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
                      [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]
            
            plt.figure(figsize=(16, 10))
            plt.imshow(self.image)
            ax = plt.gca()
            colors = COLORS * 100

            model = self.detection_model if model_type == "detection" else self.structure_model

            for score, label, (xmin, ymin, xmax, ymax), c in zip(scores.tolist(), labels.tolist(), boxes.tolist(), colors):
                ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, color=c, linewidth=1))
                try:
                    text = f'{model.config.id2label[label]}: {score:0.2f}'
                except KeyError:
                    text = f'Unknown Label: {score:0.2f}'
                ax.text(xmin, ymin, text, fontsize=15, bbox=dict(facecolor='yellow', alpha=0.5))
            
            plt.axis('off')
            plt.savefig(self.output_path, dpi=1000)
            plt.show()
        except Exception as e:
            print(f"Error during plotting or saving the results: {e}")
            raise

    def process_image(self, model_type="detection", threshold=0.7):
        """
        The main function that loads the image, processes it, detects tables, and plots the results.

        @param model_type: Specifies whether to use the detection model or structure model. Defaults to "detection".
        @param threshold: The confidence threshold for filtering results. Defaults to 0.7.
        @return: None
        """
        try:
            self.load_image()
            self.extract_features()
            outputs = self.detect_tables(model_type)
            results = self.post_process(outputs, threshold=threshold)
            self.plot_results(results, results['scores'], results['labels'], results['boxes'], model_type)
        except Exception as e:
            print(f"Error during image processing: {e}")
            raise



In [ ]:
image_path = "3eec9544-151_table_1.jpg"

In [ ]:
# Example usage:
detector = BorderlessTableDetection(image_path, "output.png")
# detector.process_image(model_type="detection", threshold=0.7)

In [ ]:
detector.process_image(model_type="structure", threshold=0.7)

In [ ]:
import json
import csv
from pathlib import Path

# Run structure detection and keep the raw detections
# (row/column objects are part of these detections)
detector.load_image()
detector.extract_features()
structure_outputs = detector.detect_tables(model_type="structure")
structure_results = detector.post_process(structure_outputs, threshold=0.6)

id2label = detector.structure_model.config.id2label

all_detections = []
for score, label_id, box in zip(
    structure_results["scores"].tolist(),
    structure_results["labels"].tolist(),
    structure_results["boxes"].tolist(),
):
    label = id2label.get(label_id, str(label_id))
    xmin, ymin, xmax, ymax = [float(v) for v in box]
    all_detections.append(
        {
            "label": label,
            "score": float(score),
            "bbox": [xmin, ymin, xmax, ymax],
            "bbox_xywh": [xmin, ymin, xmax - xmin, ymax - ymin],
        }
    )

# Explicit arrays
rows = [item for item in all_detections if "row" in item["label"].lower()]
columns = [item for item in all_detections if "column" in item["label"].lower()]

print(f"Detected rows: {len(rows)}")
print(f"Detected columns: {len(columns)}")

# Save to disk
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)
json_path = output_dir / "structure_rows_columns.json"
csv_path = output_dir / "structure_rows_columns.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump({"rows": rows, "columns": columns}, f, indent=2)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["type", "label", "score", "xmin", "ymin", "xmax", "ymax", "width", "height"],
    )
    writer.writeheader()

    for item_type, items in (("row", rows), ("column", columns)):
        for item in items:
            xmin, ymin, xmax, ymax = item["bbox"]
            _, _, width, height = item["bbox_xywh"]
            writer.writerow(
                {
                    "type": item_type,
                    "label": item["label"],
                    "score": item["score"],
                    "xmin": xmin,
                    "ymin": ymin,
                    "xmax": xmax,
                    "ymax": ymax,
                    "width": width,
                    "height": height,
                }
            )

print(f"Saved JSON: {json_path}")
print(f"Saved CSV: {csv_path}")

# Preview
rows[:2], columns[:2]

In [ ]:
import csv
import json
from pathlib import Path

# Optional OCR backend (recommended: pytesseract)
try:
    import pytesseract
    OCR_READY = True
except ImportError:
    OCR_READY = False

# Ensure image is loaded
if detector.image is None:
    detector.load_image()

# Keep only true table columns (exclude column-header box from column list)
table_rows = sorted(rows, key=lambda item: item["bbox"][1])
table_columns = sorted(
    [item for item in columns if item["label"].lower() == "table column"],
    key=lambda item: item["bbox"][0],
)
column_headers = [item for item in columns if "header" in item["label"].lower()]


def bbox_intersection(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    if x2 <= x1 or y2 <= y1:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def ocr_image(pil_image):
    if not OCR_READY:
        return ""
    return pytesseract.image_to_string(pil_image, config="--oem 3 --psm 6")


# Build row-wise table text while preserving column index
header_names = [f"col_{idx + 1}" for idx in range(len(table_columns))]
rows_as_dicts = []
cell_records = []

for row_index, row_item in enumerate(table_rows, start=1):
    current_row = {}
    for col_index, col_item in enumerate(table_columns, start=1):
        cell_box = bbox_intersection(row_item["bbox"], col_item["bbox"])
        if cell_box is None:
            cell_text = ""
        else:
            crop = detector.image.crop(tuple(cell_box))
            cell_text = ocr_image(crop)

        col_name = header_names[col_index - 1]
        current_row[col_name] = cell_text
        cell_records.append(
            {
                "row": row_index,
                "column": col_index,
                "bbox": cell_box,
                "text": cell_text,
            }
        )

    rows_as_dicts.append(current_row)

# If a column-header region exists, try OCR there to rename columns
if column_headers:
    header_box = column_headers[0]["bbox"]
    extracted_headers = []
    for col_item in table_columns:
        hdr_cell = bbox_intersection(header_box, col_item["bbox"])
        if hdr_cell is None:
            extracted_headers.append("")
        else:
            header_crop = detector.image.crop(tuple(hdr_cell))
            extracted_headers.append(ocr_image(header_crop))

    if any(text.strip() for text in extracted_headers):
        clean_headers = [
            (text.strip() if text.strip() else f"col_{idx + 1}")
            for idx, text in enumerate(extracted_headers)
        ]
        remapped_rows = []
        for row_dict in rows_as_dicts:
            remapped_rows.append(
                {
                    clean_headers[idx]: row_dict[header_names[idx]]
                    for idx in range(len(header_names))
                }
            )
        header_names = clean_headers
        rows_as_dicts = remapped_rows

# Save row-wise output
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)
row_json_path = output_dir / "table_text_by_row.json"
row_csv_path = output_dir / "table_text_by_row.csv"

with open(row_json_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "headers": header_names,
            "rows": rows_as_dicts,
            "cells": cell_records,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(row_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=header_names)
    writer.writeheader()
    writer.writerows(rows_as_dicts)

print(f"Rows: {len(table_rows)} | Columns: {len(table_columns)}")
print(f"Saved JSON: {row_json_path}")
print(f"Saved CSV: {row_csv_path}")
if not OCR_READY:
    print("pytesseract not installed. Install it to extract text:")
    print("  %pip install pytesseract")
    print("Also install Tesseract OCR app on Windows and set pytesseract.pytesseract.tesseract_cmd.")

rows_as_dicts[:2]